In [51]:
%pip install numpy pandas


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [52]:
import numpy as np
import pandas as pd

works = pd.DataFrame({
    '작가': ['김유정', '김유정', '김유정',
             '현진건', '현진건',
             '이상',   '이상',
             '나도향', '나도향',
             '채만식', '이효석'],
    '작품': ['봄봄', '동백꽃', '만무방',
             '운수 좋은 날', 'B사감과 러브레터',
             '날개', '권태',
             '벙어리 삼룡이', '물레방아',
             '레디메이드 인생', '메밀꽃 필 무렵'],
    '연도': [1935, 1936, 1934,
             1924, 1925,
             1936, 1937,
             1925, 1925,
             1934, np.nan],    # 이효석 '메밀꽃 필 무렵' 발표연도가 누락됨
    '글자수': [12500, 9800, 13600,
              11200, 7400,
              18300, 14100,
              16800, 14500,
              22000, 11800],
})

authors = pd.DataFrame({
    '작가': ['김유정', '현진건', '이상', '나도향', '채만식', '이효석', '염상섭'],
    '생년': [1908, 1900, 1910, 1902, 1902, 1907, 1897],
    '몰년': [1937, 1943, 1937, 1926, 1950, 1942, 1963],
})

Q1. 데이터프레임 살펴보기와 결측치 처리

(a) 기본 정보 출력

In [53]:
print(works.shape)    # 행/열 개수
print(works.dtypes)   # 각 열의 자료형
works.describe()      # 숫자 열 요약 통계

(11, 4)
작가         str
작품         str
연도     float64
글자수      int64
dtype: object


,연도,글자수
count,10.000000,11.000000
mean,1931.100000,13818.181818
std,5.546771,4080.641661
min,1924.000000,7400.000000
25%,1925.000000,11500.000000
50%,1934.000000,13600.000000
75%,1935.750000,15650.000000
max,1937.000000,22000.000000


짧은 관찰:
연도 열은 np.nan가 포함되어 있어 pandas가 정수형으로 저장할 수 없고, NaN을 표현하려면 float64로 자동 승격되지만 글자수는 결측치가 없으므로 int64로 유지된다.

(b) 결측치 점검

In [54]:
works.isna().sum()

작가     0
작품     0
연도     1
글자수    0
dtype: int64

짧은 관찰:
연도 열에 결측치가 1건(이효석 '메밀꽃 필 무렵'의 발표연도가 누락됨) 존재하고, 나머지 열은 결측치가 없다.

(c) 결측치 채우기

In [55]:
works2: pd.DataFrame = works.copy() # works를 복사하여 원본 보존

works2 = works2.fillna({'연도': 1936}) # 연도 열의 결측치를 1936으로 채움

works2['연도'] = works2['연도'].astype(int) # float64로 승격된 연도 열을 int로 되돌림

print(works2.dtypes)  # 연도가 int64인지 확인
works2.tail(3)        # 마지막 3행으로 이효석 행 확인

작가       str
작품       str
연도     int64
글자수    int64
dtype: object


,작가,작품,연도,글자수
8,나도향,물레방아,1925,14500
9,채만식,레디메이드 인생,1934,22000
10,이효석,메밀꽃 필 무렵,1936,11800


짧은 관찰:
fillna로 결측 연도를 1936으로 채운 후 astype(int)로 float64을 int64로 변환에 성공하였다. tail(3)으로 마지막 3행을 확인하면 이효석 행의 연도가 1936으로 채워지고 정수형임을 확인할 수 있다. 문제 조건대로 이후 모든 문항은 works2를 사용한다.

Q2. 인덱싱·필터링과 새 열 만들기

(a) 조건 필터링

In [56]:
after_1930: pd.DataFrame = works2[works2['연도'] >= 1930] # 1930년 이후 발표된 작품
print(after_1930)

kim_or_lee: pd.DataFrame = works2[works2['작가'].isin(['김유정', '이상'])] # 작가가 '김유정' 또는 '이상'인 작품
print(kim_or_lee)

     작가        작품    연도    글자수
0   김유정        봄봄  1935  12500
1   김유정       동백꽃  1936   9800
2   김유정       만무방  1934  13600
5    이상        날개  1936  18300
6    이상        권태  1937  14100
9   채만식  레디메이드 인생  1934  22000
10  이효석  메밀꽃 필 무렵  1936  11800
    작가   작품    연도    글자수
0  김유정   봄봄  1935  12500
1  김유정  동백꽃  1936   9800
2  김유정  만무방  1934  13600
5   이상   날개  1936  18300
6   이상   권태  1937  14100


짧은 관찰:
불 인덱싱은 조건식이 True인 행만 선택한다. isin을 사용하면 여러 값을 한 번에 필터링할 수 있기 때문에 | 연산자보다 간결하다.

(b) apply로 새 열 만들기

In [57]:
def categorize(n: int) -> str:
    if n < 10000:
        return '짧음'
    elif n <= 15000:
        return '보통'
    else:
        return '긴'

works2['분량'] = works2['글자수'].apply(categorize) # apply로 글자수 열에 함수 적용하여 분량 열 추가

works2[['작품', '글자수', '분량']]

,작품,글자수,분량
0,봄봄,12500,보통
1,동백꽃,9800,짧음
2,만무방,13600,보통
3,운수 좋은 날,11200,보통
4,B사감과 러브레터,7400,짧음
5,날개,18300,긴
6,권태,14100,보통
7,벙어리 삼룡이,16800,긴
8,물레방아,14500,보통
9,레디메이드 인생,22000,긴


짧은 관찰:
apply로 각 행의 글자수에 categorize 함수를 적용해 분량 열을 벡터화 방식으로 추가했다.

(c) 정렬

In [58]:
works2.sort_values(by=['연도', '글자수'], ascending=[True, False])

,작가,작품,연도,글자수,분량
3,현진건,운수 좋은 날,1924,11200,보통
7,나도향,벙어리 삼룡이,1925,16800,긴
8,나도향,물레방아,1925,14500,보통
4,현진건,B사감과 러브레터,1925,7400,짧음
9,채만식,레디메이드 인생,1934,22000,긴
2,김유정,만무방,1934,13600,보통
0,김유정,봄봄,1935,12500,보통
5,이상,날개,1936,18300,긴
10,이효석,메밀꽃 필 무렵,1936,11800,보통
1,김유정,동백꽃,1936,9800,짧음


짧은 관찰:
sort_values의 by와 ascending에 리스트를 넘겨 다중 기준 정렬을 한 줄로 처리하였다. 연도가 같은 경우 글자수가 많은 작품이 먼저 출력된다.

Q3. groupby로 그룹별 집계

(a) 작가별 평균·작품 수·총 글자수

In [59]:
# 작가별 글자수 통계를 한 번의 groupby + agg로 계산
result_a: pd.DataFrame = works2.groupby('작가')['글자수'].agg(['mean', 'count', 'sum'])

# 열 이름을 알아보기 쉽게 변경
result_a.columns = ['평균 글자수', '작품 수', '총 글자수']
result_a

,평균 글자수,작품 수,총 글자수
작가,,,
김유정,11966.666667,3,35900
나도향,15650.000000,2,31300
이상,16200.000000,2,32400
이효석,11800.000000,1,11800
채만식,22000.000000,1,22000
현진건,9300.000000,2,18600


짧은 관찰:
groupby로 작가별로 묶은 뒤 agg에 집계 함수 리스트를 넘겨 평균, 작품 수, 총 글자수를 한 번에 계산하였다. for문 없이 벡터화 연산으로 처리된다.

(b) 분량별 작품 수

In [60]:
result_b: pd.Series = works2['분량'].value_counts() # 분량 열의 빈도를 value_counts로 집계
result_b

분량
보통    6
긴     3
짧음    2
Name: count, dtype: int64

짧은 관찰:
value_counts는 각 고유값의 등장 횟수를 내림차순으로 반환하여 별도의 groupby 없이 단일 열의 빈도를 간결하게 구할 수 있다.

(c) 작가별·분량별 작품 수

In [61]:
result_c: pd.Series = works2.groupby(['작가', '분량']).size() # 작가와 분량의 모든 조합에 대한 작품 수 집계
result_c

작가   분량
김유정  보통    2
     짧음    1
나도향  긴     1
     보통    1
이상   긴     1
     보통    1
이효석  보통    1
채만식  긴     1
현진건  보통    1
     짧음    1
dtype: int64

관찰:
groupby(['작가', '분량']).size()로 두 열의 조합별 작품 수를 한 번에 구했고, 가장 많은 작품 수를 가진 조합은 '김유정–보통(2편)'으로, 김유정이 1만에서 1만 5천 자 내외의 중편 분량을 선호했음을 시사한다.

Q4. merge로 두 표 합치기

(a) left merge

In [62]:
# works2를 왼쪽 기준으로 authors와 '작가' 키로 left merge
full: pd.DataFrame = pd.merge(works2, authors, on='작가', how='left')

print(full.shape)  # 합쳐진 표의 shape 확인
full

(11, 7)


,작가,작품,연도,글자수,분량,생년,몰년
0,김유정,봄봄,1935,12500,보통,1908,1937
1,김유정,동백꽃,1936,9800,짧음,1908,1937
2,김유정,만무방,1934,13600,보통,1908,1937
3,현진건,운수 좋은 날,1924,11200,보통,1900,1943
4,현진건,B사감과 러브레터,1925,7400,짧음,1900,1943
5,이상,날개,1936,18300,긴,1910,1937
6,이상,권태,1937,14100,보통,1910,1937
7,나도향,벙어리 삼룡이,1925,16800,긴,1902,1926
8,나도향,물레방아,1925,14500,보통,1902,1926
9,채만식,레디메이드 인생,1934,22000,긴,1902,1950


확인 질문:
how='left'는 왼쪽 DataFrame(works2)의 모든 행을 유지하고 오른쪽(authors)에서 일치하는 행을 붙이는데, left merge는 왼쪽 기준이므로 오른쪽에만 존재하는 행은 버려지기 때문에 authors에만 있고 works2에 없는 작가(염상섭)는 포함되지 않는다.

(b) 파생 열 집필 당시 나이

In [63]:
full['집필 당시 나이'] = full['연도'] - full['생년'] # 집필 당시 나이 = 연도 - 생년

full[['작가', '작품', '연도', '생년', '집필 당시 나이']] # 관련 열만 선택하여 출력

,작가,작품,연도,생년,집필 당시 나이
0,김유정,봄봄,1935,1908,27
1,김유정,동백꽃,1936,1908,28
2,김유정,만무방,1934,1908,26
3,현진건,운수 좋은 날,1924,1900,24
4,현진건,B사감과 러브레터,1925,1900,25
5,이상,날개,1936,1910,26
6,이상,권태,1937,1910,27
7,나도향,벙어리 삼룡이,1925,1902,23
8,나도향,물레방아,1925,1902,23
9,채만식,레디메이드 인생,1934,1902,32


짧은 관찰:
연도와 생년 열을 벡터화 빼기 연산을 통해 for문 없이 모든 행의 집필 당시 나이를 한 번에 계산했다.

(c)  작가별 집필 평균 나이

In [64]:
# 작가별 집필 평균 나이를 groupby로 계산 후 오름차순 정렬
result_c: pd.Series = (
    full.groupby('작가')['집필 당시 나이']
    .mean()
    .sort_values(ascending=True)
)
result_c

작가
나도향    23.0
현진건    24.5
이상     26.5
김유정    27.0
이효석    29.0
채만식    32.0
Name: 집필 당시 나이, dtype: float64

해석:
가장 어린 나이에 작품을 발표한 작가는 나도향(평균 23세)이며, 가장 늦은 나이에 작품을 발표한 작가는 채만식(평균 32세)이다. 전반적으로 1920에서 1930년대 작가들은 20대 초반 ~ 30대에 주요 작품을 집필했음을 알 수 있다.

생성형 AI 참조 링크: https://claude.ai/share/743dd24e-19f8-4f32-8e4d-75e8269a334a